In [1]:
# In[1]:


# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print((os.path.join(dirname, filename)))

# Any results you write to the current directory are saved as output.

In [2]:
# In[2]:


from sklearn.datasets import load_wine

In [3]:
# In[3]:


wine_data = load_wine()

In [4]:
# In[4]:


data = pd.DataFrame(wine_data["data"], columns=wine_data["feature_names"])
data["Target"] = wine_data["target"]

In [5]:
# In[5]:


data.head()

In [6]:
# In[6]:


from sklearn.tree import DecisionTreeClassifier

In [ ]:
# In[7]:


from sklearn.model_selection import train_test_split

x = data.drop("Target", axis=1)
y = data["Target"]

# Hold out a test set before any fitting/tuning so final evaluation is on unseen data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
# In[8]:


clf = DecisionTreeClassifier(random_state=42)

In [ ]:
# In[9]:


clf.fit(x_train, y_train)

In [ ]:
# In[10]:


# Predict a held-out sample (not seen during training)
sample_idx = x_test.index[0]
print((clf.predict(x_test.loc[[sample_idx]])))
print((data.loc[sample_idx]))
print((clf.score(x_test, y_test)))

In [11]:
# In[11]:


from sklearn.model_selection import cross_val_score

In [ ]:
# In[12]:


clf = DecisionTreeClassifier(random_state=42, max_depth=3, max_features=3)
cross_val = cross_val_score(clf, x_train, y_train, cv=5)
print((cross_val.mean()))

In [13]:
# In[13]:


print(cross_val)

In [ ]:
# In[14]:


from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
# In[15]:


# Fit the scaler on the training split only; inside CV, scaling is done via a pipeline
skl = StandardScaler()
skl.fit(x_train)
x_train_scaled = skl.transform(x_train)
x_test_scaled = skl.transform(x_test)

In [ ]:
# In[16]:


clf = DecisionTreeClassifier(random_state=42)
model = cross_val_score(clf, x_train, y_train, cv=5)
print((model.mean()))
model = cross_val_score(make_pipeline(StandardScaler(), clf), x_train, y_train, cv=5)
print((model.mean()))

In [18]:
# In[18]:


from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold

In [ ]:
# In[19]:


score = []
for i in range(1,80):
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=i))
    cv = KFold(random_state=42, n_splits=5, shuffle=True)
    res = cross_val_score(knn, x_train, y_train, cv=cv)
    score.append(res.mean())

In [20]:
# In[20]:


import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# In[21]:


plt.plot(score);
plt.grid()
print((max(score)))
best_k = score.index(max(score))+1
print(best_k)

In [ ]:
# In[22]:


knn = KNeighborsClassifier(n_neighbors=best_k)
res = cross_val_score(make_pipeline(StandardScaler(), knn), x_train, y_train, cv=5)
print((res.mean()))

In [ ]:
# In[23]:


knn.fit(x_train_scaled, y_train);
print((knn.predict([x_test_scaled[0]])))
# Final evaluation on the held-out test set
print((knn.score(x_test_scaled, y_test)))